# Titanic: Predicting Survival

## Question

Can a model predict which passengers survived the Titanic, using only what was
known about them before the voyage ended (class, sex, age, family and so on)?
And how much better is it than guessing?

## Data

- Source: Kaggle's [Titanic - Machine Learning from Disaster](https://www.kaggle.com/c/titanic)
  (`train.csv`, 891 passengers).
- The raw file was loaded, cleaned and explored in MySQL first (see
  `titanic_setup.sql` and `titanic_eda.sql`).
- The features used here come from a MySQL view, `train_features`, exported to
  `titanic_features.csv` (13 columns). It adds `Title` (from the name),
  `FamilySize`, `IsAlone` and `HasCabin`.

## Steps

1. Load the CSV and check its shape and missing values.
2. Fix a data problem: the export turned missing ages into zeros, so convert
   them back to missing.
3. Separate the target (`Survived`) from the features.
4. Split into training and test sets, before any filling of missing values.
5. Build a baseline that always predicts the most common outcome.
6. Build a pipeline: fill missing values, scale the numbers, one-hot encode the
   text columns, then fit a logistic regression.
7. Evaluate on the test set with accuracy and a confusion matrix.
8. Cross-validate (5 folds) for a more reliable score.
9. Run comparison experiments: without `HasCabin`, a random forest, and filling
   age by title instead of the overall median.

## Notes

- All fill values are learned from the training rows only, inside the pipeline,
  so no information from the test passengers leaks into training.
- Differences smaller than about 0.025 between scores are treated as noise,
  because that's the fold-to-fold spread.
  

## Results

- Dummy baseline (always predict "did not survive"): 61.5% accuracy.
- Logistic regression: 83.2% on the held-out 20% (179 passengers), and
  82.5% in 5-fold cross-validation (std 0.025). I'd quote the cross-validated
  figure as the main result.
- On the held-out split the model found about 88% of the passengers who died
  and 75% of the survivors (confusion matrix: [[97, 13], [17, 52]]).
- Removing `HasCabin`: 82.2%, so the model doesn't depend on it.
- Random forest: 80.2%, lower than logistic regression by about the size of
  the fold-to-fold variation, so I can't say it is truly worse.
- Filling age by title instead of the overall median: 82.7%, which isn't
  distinguishable from the plain model.

**Takeaway:** with these engineered features, a simple logistic regression
performs about as well as the more complex options I tried.

## Data note

The CSV exported from MySQL had turned missing ages into zeros. I caught it
when the age-fill step reported 0 missing values, and fixed it by converting
zeros back to missing before the split. The scores above are from after the
fix.

In [41]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

In [21]:
df =pd.read_csv("titanic_features.csv")
df

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,HasCabin
0,1,0,3,male,22.0,1,0,7.2500,S,Mr,2,0,0
1,2,1,1,female,38.0,1,0,71.2833,C,Mrs,2,0,1
2,3,1,3,female,26.0,0,0,7.9250,S,Miss,1,1,0
3,4,1,1,female,35.0,1,0,53.1000,S,Mrs,2,0,1
4,5,0,3,male,35.0,0,0,8.0500,S,Mr,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.0,0,0,13.0000,S,Other,1,1,0
887,888,1,1,female,19.0,0,0,30.0000,S,Miss,1,1,1
888,889,0,3,female,0.0,1,2,23.4500,S,Miss,4,0,0
889,890,1,1,male,26.0,0,0,30.0000,C,Mr,1,1,1


In [22]:
import numpy as  np
df["Age"] = df["Age"].replace(0,np.nan)
print(df["Age"].isna().sum())

177


In [23]:
df.shape
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         2
Title            0
FamilySize       0
IsAlone          0
HasCabin         0
dtype: int64

In [24]:
from sklearn.model_selection import train_test_split

X= df.drop(columns =["Survived","PassengerId"])
y = df["Survived"]

X_train,X_test,y_train,y_test = train_test_split(X,y, test_size =0.2,random_state= 42, stratify= y)
print(X_train.shape,X_test.shape)

(712, 11) (179, 11)


In [25]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train,y_train)

# accuracy_score(y_true,y_pred)


print("Baseline accuracy:" ,accuracy_score(y_test,dummy.predict(X_test)))

Baseline accuracy: 0.6145251396648045


In [26]:
# The Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression

num_cols = ["Age","SibSp","Parch","Fare","FamilySize","IsAlone","HasCabin","Pclass"]
cat_cols = ["Sex","Embarked","Title"]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy = "median")),
    ("scale", StandardScaler() ),
])
cat_pipe = Pipeline([
    ("impute",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore")),
])

prep = ColumnTransformer([
    ("num", num_pipe,num_cols),
    ("cat",cat_pipe,cat_cols),
])

model = Pipeline([
    ("prep",prep),
    ("clf", LogisticRegression(max_iter = 1000)),
])
model.fit(X_train,y_train)
pred = model.predict(X_test)
print("Model accuracy: ", accuracy_score(y_test, pred))



Model accuracy:  0.8324022346368715


In [27]:
# A look beyond accuracy
from sklearn.metrics import confusion_matrix,classification_report

print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))

[[97 13]
 [17 52]]
              precision    recall  f1-score   support

           0       0.85      0.88      0.87       110
           1       0.80      0.75      0.78        69

    accuracy                           0.83       179
   macro avg       0.83      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



In [28]:
# Check to ensure the results are more trustworthy
# 1.Cross-valdation

from  sklearn.model_selection import cross_val_score
scores = cross_val_score(model,X,y, cv =5, scoring="accuracy")
print(scores)
print("Mean", scores.mean(),"Std",scores.std())

[0.83240223 0.80337079 0.80337079 0.81460674 0.87078652]
Mean 0.8249074132195091 Std 0.025277740857140973


In [29]:
num_cols_no_cabin = [c for c in num_cols if c != "HasCabin"]

In [30]:
# That shows raw rows as the model receives them
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone,HasCabin
692,3,male,NaN,0,0,56.4958,S,Mr,1,1,0
481,2,male,NaN,0,0,0.0000,S,Mr,1,1,0
527,1,male,NaN,0,0,221.7792,S,Mr,1,1,1
855,3,female,18.0,0,1,9.3500,S,Mrs,2,0,0
801,2,female,31.0,1,1,26.2500,S,Mrs,3,0,0


In [31]:
prep.fit(X_train)
print(prep.transform(X_train).shape)

(712, 18)


In [32]:
print(model.named_steps["clf"].coef_)

[[-0.39818153 -0.32507556 -0.1453585   0.10994225 -0.29366283 -0.16507707
   0.38907407 -0.59337355  0.80890467 -0.75577448  0.02175167  0.32072982
  -0.28935129  1.27998476 -0.10241071 -1.26619075  0.58279581 -0.44104892]]


In [33]:
# Helper function that builds the pipeline
from sklearn.ensemble import RandomForestClassifier

def make_model(num_cols,cat_cols,clf):
    num_pipe = Pipeline([
        ("impute",SimpleImputer(strategy="median")),
        ("scale",StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("impute",SimpleImputer(strategy ="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore")),
    ])
    prep = ColumnTransformer([
        ("num",num_pipe,num_cols),
        ("cat",cat_pipe,cat_cols),
    ])
    return Pipeline([("prep",prep),("clf",clf)])

In [34]:
# Three experiments,  scored the same way

num_no_cabin = [c for c in num_cols if c != "HasCabin"]

experiments ={
    "Logistic,all features": make_model(num_cols,cat_cols,LogisticRegression(max_iter = 1000)),
    "Logistic, no HasCabin": make_model(num_no_cabin,cat_cols,LogisticRegression(max_iter=1000)),
    "Random forest, all features": make_model(num_cols,cat_cols,RandomForestClassifier(n_estimators=300,random_state=42)),

}

for name, m in experiments.items():
    s = cross_val_score(m,X,y,cv = 5,scoring="accuracy")
    print(f"{name}: mean {s.mean():.3f}, std {s.std():.3f}")

Logistic,all features: mean 0.825, std 0.025
Logistic, no HasCabin: mean 0.822, std 0.023
Random forest, all features: mean 0.802, std 0.031


In [35]:
# Age-by_title step
from sklearn.base import BaseEstimator,TransformerMixin

class AgeByTitle(BaseEstimator,TransformerMixin):
    def fit(self,X,y= None):
        self.medians_ = X.groupby("Title")["Age"].median()
        self.overall_ = X["Age"].median()
        return self

    def transform(self,X):
        X = X.copy()
        fill = X["Title"].map(self.medians_).fillna(self.overall_)
        X["Age"] = X["Age"].fillna(fill)
        return X

age_model = Pipeline([
    ("age", AgeByTitle()),
    ("rest", make_model(num_cols,cat_cols,LogisticRegression(max_iter=1000))),
])

s = cross_val_score(age_model,X,y,cv = 5,scoring ="accuracy")
print(f"Logistic, age by title: mean {s.mean():.3f}, std {s.std():.3f}")

Logistic, age by title: mean 0.827, std 0.026


In [36]:
step = AgeByTitle().fit(X_train)
print(step.medians_)
filled = step.transform(X_train)
print(X_train["Age"].isna().sum(),"->",filled["Age"].isna().sum())

Title
Master     3.0
Miss      22.0
Mr        30.0
Mrs       35.0
Other     49.0
Name: Age, dtype: float64
137 -> 0


In [37]:
print(df["Age"].isna().sum())
print((df["Age"]==0).sum())
print(df["Age"].describe())

177
0
count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64


In [38]:
print(df["Embarked"].value_counts(dropna=False))
print(df.isna().sum())


Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64
PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         2
Title            0
FamilySize       0
IsAlone          0
HasCabin         0
dtype: int64
